# Lesson 9: Low-rank adaptation (LoRA)

Freeze a small pretrained model and learn a low-rank update to its output projection.

**How to run:** Select a Python kernel with PyTorch installed, then run each code cell from top to bottom with **Shift+Enter**. This notebook is self-contained; no other notebook needs to run first. Restart the kernel and run from the top to reset the experiment.

**Source:** This lesson was developed from the [reference conversation's roadmap](https://chatgpt.com/share/6aa56bca-4a1c-83e9-9153-1edcc7ff7e40). The reference supplies Lesson 1 and a topic outline; Lessons 2–12 are newly written implementations of those topics. Small examples demonstrate the mechanics; they are not trained assistants.


In [ ]:
import math
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
# Small tensors can be slower with many CPU threads.
torch.set_num_threads(1)
device = torch.device('cpu')
print('PyTorch:', torch.__version__, '| device:', device)


## Load and tokenize text

We reuse `input.txt`, resolving it from the notebook folder or repository root. `B` means batch size, `T` means context length, and `C` means vector width. Our file is only 80 characters, so the validation scores are noisy and text generation will be limited.


In [ ]:
input_path = Path('input.txt')
if not input_path.is_file():
    input_path = Path('chatgpt/input.txt')
text = input_path.read_text(encoding='utf-8')
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return ''.join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.8 * len(data))
train_data, val_data = data[:split], data[split:]
block_size = min(8, len(train_data) - 1, len(val_data) - 1)
if block_size < 1:
    raise ValueError('input.txt needs enough text for train and validation sequences.')
batch_size = 4
print('Characters:', vocab_size, '| train:', len(train_data), '| validation:', len(val_data))
print('Context length:', block_size)


## Draw random batches

Choose starting positions within one split. Targets are the same window shifted right by one token. No window crosses from training into validation. The tokenizer vocabulary uses the full text so every validation character has an ID; model weights are updated only on training tokens.


In [ ]:
def get_batch(split_name='train'):
    if split_name not in ('train', 'val'):
        raise ValueError("Choose 'train' or 'val'.")
    source = train_data if split_name == 'train' else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in starts])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch()
print('Input shape:', xb.shape, '| target shape:', yb.shape)
print('Input :', repr(decode(xb[0])))
print('Target:', repr(decode(yb[0])))
assert torch.equal(xb[:, 1:], yb[:, :-1])


## Causal multi-head attention

Each head compares queries to keys, then combines value vectors. A triangular mask prevents reading future tokens. This is the implementation developed in Lessons 3–4, included here so this notebook runs independently.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        assert width % heads == 0
        self.heads = heads
        self.head_size = width // heads
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.projection = nn.Linear(width, width)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('causal_mask', torch.tril(torch.ones(context_length, context_length, dtype=torch.bool)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # Give each head its own vector slice: [B, heads, T, head_size].
        q = q.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        k = k.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        v = v.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_size)
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float('-inf'))
        weights = self.dropout(F.softmax(scores, dim=-1))
        out = (weights @ v).transpose(1, 2).contiguous().reshape(B, T, C)
        return self.projection(out)


## Transformer block

LayerNorm normalizes each token's features. Residual additions let information pass around attention and the MLP. The MLP expands each token vector, applies a nonlinear function, and projects it back.


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(width)
        self.attention = CausalSelfAttention(width, heads, context_length, dropout)
        self.ln2 = nn.LayerNorm(width)
        self.mlp = nn.Sequential(
            nn.Linear(width, 4 * width), nn.GELU(),
            nn.Linear(4 * width, width), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


## Token and position embeddings → vocabulary scores

Token embeddings describe characters; learned position embeddings distinguish their positions. The final linear layer predicts the next character at every position. Cross entropy consumes raw scores (logits).


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, context_length, width=32, heads=4, layers=2):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(context_length, width)
        self.blocks = nn.Sequential(*[
            TransformerBlock(width, heads, context_length) for _ in range(layers)
        ])
        self.final_norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.context_length:
            raise ValueError('Sequence exceeds context length.')
        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        logits = self.lm_head(self.final_norm(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

model = TinyGPT(vocab_size, block_size).to(device)
print('Parameters:', sum(p.numel() for p in model.parameters()))


## Train a small model

This cell continues training if rerun. Rerun the model-creation cell first for a fresh model. The small default run teaches the workflow; useful generation needs substantially more text and training.


In [ ]:
training_steps = 120
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
model.train()
for step in range(training_steps):
    x, y = get_batch()
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 40 == 0 or step == training_steps - 1:
        print(f'step {step}: loss {loss.item():.4f}')


## Replace one linear layer with a LoRA wrapper

For a linear layer, LoRA learns `W + (alpha/r) B A` while keeping `W` frozen. `A` maps inputs to a small rank-r vector and `B` maps back to outputs. Initializing `B` to zero preserves the original predictions. We adapt only the output projection to keep this example small; attention projections are common adapter targets too.


In [ ]:
import copy

class LoRALinear(nn.Module):
    def __init__(self, base, rank=4, alpha=8):
        super().__init__()
        self.base = base
        self.scale = alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.in_features, device=base.weight.device))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank, device=base.weight.device))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        for parameter in self.base.parameters():
            parameter.requires_grad_(False)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale

adapted = copy.deepcopy(model)
for parameter in adapted.parameters():
    parameter.requires_grad_(False)
adapted.lm_head = LoRALinear(adapted.lm_head)
model.eval()
adapted.eval()
probe, _ = get_batch()
with torch.no_grad():
    assert torch.allclose(model(probe)[0], adapted(probe)[0])
print('Trainable adapter parameters:', sum(p.numel() for p in adapted.parameters() if p.requires_grad))
print('Total parameters:', sum(p.numel() for p in adapted.parameters()))


## Adapt on a small domain sample

The adaptation text uses the existing tokenizer vocabulary. A frozen tokenizer cannot represent new characters without a separate vocabulary/model change. We save copies of frozen parameters to verify that only the adapters change.


In [ ]:
adaptation_text = text[:max(block_size + 1, len(text) // 2)] * 4
adaptation_ids = torch.tensor(encode(adaptation_text), dtype=torch.long, device=device)
frozen_before = {name: p.detach().clone() for name, p in adapted.named_parameters() if not p.requires_grad}
optimizer = torch.optim.AdamW([p for p in adapted.parameters() if p.requires_grad], lr=1e-2)
adapted.train()
for step in range(60):
    starts = torch.randint(len(adaptation_ids) - block_size, (4,)).tolist()
    x = torch.stack([adaptation_ids[i:i + block_size] for i in starts])
    y = torch.stack([adaptation_ids[i + 1:i + block_size + 1] for i in starts])
    _, loss = adapted(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print(step, loss.item())
for name, parameter in adapted.named_parameters():
    if name in frozen_before:
        assert torch.equal(parameter, frozen_before[name]), name
print('All frozen parameters stayed unchanged.')


## Merge the adapter for inference

The merged weight adds the low-rank update to the original weight. Check that the merged and wrapped layers produce the same outputs, allowing for floating-point rounding.


In [ ]:
adapted.eval()
merged = copy.deepcopy(adapted)
wrapper = merged.lm_head
with torch.no_grad():
    wrapper.base.weight.add_(wrapper.scale * (wrapper.B @ wrapper.A))
merged.lm_head = wrapper.base
with torch.no_grad():
    a = adapted(probe)[0]
    b = merged(probe)[0]
assert torch.allclose(a, b, atol=1e-5)
print('Maximum merge difference:', (a - b).abs().max().item())


## Try it yourself

Compare ranks 1, 4, and 8 from fresh base models. Compute the adapter parameter count as `rank * (input_width + output_width)`. Explain why LoRA reduces trainable state but still requires the frozen base weights.
